In [10]:
# !pip install -q datasets pandas sentence-transformers pymilvus

# # Clone the GitHub prompt dataset
# !git clone https://github.com/f/awesome-chatgpt-prompts.git

# import pandas as pd

# df = pd.read_csv("awesome-chatgpt-prompts/prompts.csv")

# # Combine 'act' and 'prompt' for better context
# df['full_prompt'] = df['act'] + " - " + df['prompt']

# df[['full_prompt']].head()

In [11]:
# from sentence_transformers import SentenceTransformer
# import numpy as np

# # Load the embedding model
# model = SentenceTransformer('all-MiniLM-L6-v2')

# # Generate vector embeddings from the prompts
# embeddings = model.encode(df['full_prompt'].tolist(), show_progress_bar=True)
# embeddings = np.array(embeddings)

# print("✅ Embeddings generated:", embeddings.shape)

In [ ]:
# from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility

# ZILLIZ_URI = "https://in03-82999feec6d34ff.serverless.gcp-us-west1.cloud.zilliz.com"
# TOKEN = "b4ba8ba6e959eb24aff32b920a896b9da8545bf2497f2048479ee872ba0cc5680d74b3be1c3941dd79b35ef4a0d94602778ea0f4"

# connections.connect(uri=ZILLIZ_URI, token=TOKEN)
# print("✅ Connected to Milvus!")

# # Define schema with auto-generated primary ID
# collection_name = "prompt_builder_agent"

# fields = [
#     FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
#     FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=embeddings.shape[1]),
#     FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=1024),
# ]

# schema = CollectionSchema(fields, description="Prompt builder collection with auto ID")

# if not utility.has_collection(collection_name):
#     collection = Collection(name=collection_name, schema=schema)
#     print(f"✅ Collection `{collection_name}` created!")
# else:
#     collection = Collection(name=collection_name)
#     print(f"✅ Collection `{collection_name}` already exists!")


In [2]:
pip install pymilvus

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.71.0
    Uninstalling grpcio-1.71.0:
      Successfully uninstalled grpcio-1.71.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.0 requires grpcio>=1.71.0, but you have grpcio 1.67.1 which is incompatible.


In [3]:
pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 41.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
# Milvus connection details
ZILLIZ_URI = "https://in03-82999feec6d34ff.serverless.gcp-us-west1.cloud.zilliz.com"
TOKEN = "b4ba8ba6e959eb24aff32b920a896b9da8545bf2497f2048479ee872ba0cc5680d74b3be1c3941dd79b35ef4a0d94602778ea0f4"
COLLECTION_NAME = "prompt_builder_agent"

# Connect to Milvus
print("Connecting to Milvus...")
connections.connect(uri=ZILLIZ_URI, token=TOKEN)
print("✅ Connected to Milvus!")

# Model paths
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

Connecting to Milvus...
✅ Connected to Milvus!


In [3]:
def load_prompts_data():
    """Load and preprocess prompts from the Awesome ChatGPT Prompts repository"""
    try:
        # Clone the repository (only if it doesn't exist)
        import os
        if not os.path.exists("awesome-chatgpt-prompts"):
            !git clone https://github.com/f/awesome-chatgpt-prompts.git

        # Load the CSV data
        df = pd.read_csv("awesome-chatgpt-prompts/prompts.csv")

        # Combine 'act' and 'prompt' for better context
        df['full_prompt'] = df['act'] + " - " + df['prompt']

        print(f"✅ Loaded {len(df)} prompts from the dataset")
        return df
    except Exception as e:
        print(f"❌ Failed to load prompts data: {str(e)}")
        return None

In [4]:
def setup_vector_database(df):
    """Set up the vector database with embeddings from the prompts dataset"""
    try:
        # Load embedding model
        print("Loading embedding model...")
        model = SentenceTransformer(EMBEDDING_MODEL)

        # Generate embeddings
        print("Generating embeddings (this may take a while)...")
        embeddings = model.encode(df['full_prompt'].tolist(), show_progress_bar=True)
        embeddings = np.array(embeddings)
        print(f"✅ Generated {len(embeddings)} embeddings of dimension {embeddings.shape[1]}")

        # Check if collection exists and recreate it
        if utility.has_collection(COLLECTION_NAME):
            utility.drop_collection(COLLECTION_NAME)
            print(f"✅ Dropped existing collection: {COLLECTION_NAME}")

        # Define schema with explicit IDs
        fields = [
            FieldSchema(name="id", dtype=DataType.INT64, is_primary=True),
            FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=embeddings.shape[1]),
            FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=5000),
        ]
        schema = CollectionSchema(fields, description="Prompt builder collection")
        collection = Collection(name=COLLECTION_NAME, schema=schema)
        print(f"✅ Created collection: {COLLECTION_NAME}")

        # Insert data with explicit IDs
        entities = [
            list(range(1, len(df) + 1)),
            embeddings.tolist(),
            df['full_prompt'].tolist()
        ]

        collection.insert(entities)
        print(f"✅ Inserted {len(df)} records into the collection")

        # Create an index for the vector field
        index_params = {
            "metric_type": "COSINE",
            "index_type": "HNSW",
            "params": {"M": 8, "efConstruction": 64}
        }

        collection.create_index("vector", index_params)
        print("✅ Created vector index")

        # Load the collection for searching
        collection.load()
        print("✅ Collection loaded and ready for querying")

        return True
    except Exception as e:
        print(f"❌ Failed to set up vector database: {str(e)}")
        return False

In [5]:
def search_prompts(query, top_k=5):
    """Search for similar prompts using vector similarity"""
    try:
        # Connect to the database
        collection = Collection(name=COLLECTION_NAME)
        collection.load()

        # Generate embedding for the query
        model = SentenceTransformer(EMBEDDING_MODEL)
        query_embedding = model.encode([query])[0].tolist()

        # Define search parameters
        search_params = {"metric_type": "COSINE", "params": {"ef": 32}}

        # Execute the search
        results = collection.search(
            data=[query_embedding],
            anns_field="vector",
            param=search_params,
            limit=top_k,
            output_fields=["text"]
        )

        # Return the search results
        return [hit.entity.get('text') for hit in results[0]]
    except Exception as e:
        print(f"❌ Search failed: {str(e)}")
        return []

def analyze_prompts(prompts):
    """Analyze a list of prompts to extract patterns and key components"""
    analysis = {
        "roles": [],        # The primary role/persona in each prompt
        "instructions": [], # The main instructions in each prompt
    }

    for prompt in prompts:
        # Split by the common format "ROLE - INSTRUCTIONS"
        parts = prompt.split(" - ", 1)

        if len(parts) > 1:
            role = parts[0]
            instructions = parts[1]
        else:
            if "act as" in prompt.lower():
                role_start = prompt.lower().find("act as") + 7
                role_end = prompt.find(".", role_start)
                if role_end == -1:  # No period found
                    role_end = prompt.find("\n", role_start)
                if role_end == -1:  # No newline found
                    role_end = len(prompt)

                role = prompt[role_start:role_end].strip()
                instructions = prompt
            else:
                role = "Expert"
                instructions = prompt

        role = role.replace("I want you to act as a", "").replace("I want you to act as an", "")
        role = role.replace("I want you to act as", "").strip()

        analysis["roles"].append(role)
        analysis["instructions"].append(instructions)

    return analysis

def extract_key_terms(text):
    """Extract key terms from a text using simple text processing"""
    # Simple word tokenization
    words = text.lower().split()

    # Common English stopwords
    stopwords = {
        "a", "an", "the", "and", "or", "but", "if", "then", "else", "when",
        "at", "by", "for", "with", "about", "against", "between", "into",
        "through", "during", "before", "after", "above", "below", "to", "from",
        "up", "down", "in", "out", "on", "off", "over", "under", "again",
        "further", "then", "once", "here", "there", "when", "where", "why",
        "how", "all", "any", "both", "each", "few", "more", "most", "other",
        "some", "such", "no", "nor", "not", "only", "own", "same", "so",
        "than", "too", "very", "s", "t", "can", "will", "just", "don", "should",
        "now", "d", "ll", "m", "o", "re", "ve", "y", "ain", "aren", "couldn",
        "didn", "doesn", "hadn", "hasn", "haven", "isn", "ma", "mightn", "mustn",
        "needn", "shan", "shouldn", "wasn", "weren", "won", "wouldn", "i", "me",
        "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours"
    }

    # Filter out stopwords and short words, remove punctuation
    filtered_words = []
    for word in words:
        word = word.strip(".,!?;:()[]{}\"'")
        if word not in stopwords and len(word) > 2:
            filtered_words.append(word)

    return filtered_words

def find_best_match(user_query, prompts):
    """Find the best matching prompt based on the user query"""
    query_terms = extract_key_terms(user_query)

    # Score each prompt based on term overlap
    scores = []
    for prompt in prompts:
        prompt_terms = extract_key_terms(prompt)

        # Calculate overlap
        term_overlap = sum(1 for term in query_terms if term in prompt_terms)
        scores.append(term_overlap)

    # Return index of prompt with highest score
    if scores:
        return scores.index(max(scores))
    return 0

def extract_user_requirements(user_query):
    """Extract specific requirements from user query using simple text matching"""
    requirements = {
        "domain": "",       # Specific field or domain
        "constraints": [],  # Any limitations or constraints
    }

    # Check for domain indicators
    domain_indicators = ["about", "for", "in the field of", "related to", "on the topic of"]
    for indicator in domain_indicators:
        if indicator in user_query.lower():
            parts = user_query.lower().split(indicator, 1)
            if len(parts) > 1:
                domain = parts[1].split()[0:5]  # Take first few words after indicator
                requirements["domain"] = " ".join(domain).strip(".,!?;:()")
                break

    # Check for constraints
    constraint_phrases = ["make sure", "ensure that", "must be", "should be", "needs to be"]
    for phrase in constraint_phrases:
        if phrase in user_query.lower():
            parts = user_query.lower().split(phrase, 1)
            if len(parts) > 1:
                constraint = parts[1].split(".")[0]  # Take text until next period
                requirements["constraints"].append(constraint.strip())

    return requirements


In [6]:
def generate_custom_prompt(user_query, top_k=5):
    """Generate a customized prompt based on user query"""
    # Retrieve relevant prompts
    similar_prompts = search_prompts(user_query, top_k=top_k)

    if not similar_prompts:
        return "I couldn't find relevant prompts for your request. Please try a different query."

    # Analyze the retrieved prompts
    analysis = analyze_prompts(similar_prompts)

    # Find the best matching prompt
    best_idx = find_best_match(user_query, similar_prompts)

    # Extract user requirements
    requirements = extract_user_requirements(user_query)

    # Build the custom prompt
    role = analysis["roles"][best_idx] if analysis["roles"] else "Expert"

    # Format in the style of the dataset
    custom_prompt = f"I want you to act as a {role}. "

    # Add core instructions from best matching prompt
    if analysis["instructions"]:
        instructions = analysis["instructions"][best_idx]
        # Remove any request examples to avoid confusion
        for marker in ["My first request is", "My request is", "For example"]:
            if marker in instructions:
                instructions = instructions.split(marker)[0].strip()
        custom_prompt += instructions

    # Add user-specific requirements
    if requirements["domain"]:
        custom_prompt += f" Focus specifically on {requirements['domain']}."

    for constraint in requirements["constraints"]:
        custom_prompt += f" Make sure {constraint}."

    # Add a request placeholder that matches the style of the dataset
    custom_prompt += " My request is: " + user_query

    return custom_prompt

def enhance_prompt_with_tinyllama(base_prompt):
    """Enhance a prompt using the TinyLlama model"""
    try:
        # Load TinyLlama model and tokenizer
        tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
        model = AutoModelForCausalLM.from_pretrained(LLM_MODEL)

        # Move model to GPU if available
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)

        # Prepare input
        prompt = f"Improve the following prompt for clarity and creativity:\n\n{base_prompt}\n\nEnhanced prompt:"
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        # Generate enhanced prompt
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode and extract enhanced prompt
        result = tokenizer.decode(outputs[0], skip_special_tokens=True)
        enhanced = result.split("Enhanced prompt:")[-1].strip()

        return enhanced
    except Exception as e:
        print(f"❌ TinyLlama enhancement failed: {str(e)}")
        return base_prompt  # Fallback to base prompt

def generate_final_prompt(user_query):
    """Generate a final prompt by customizing and enhancing it"""
    base_prompt = generate_custom_prompt(user_query)
    return enhance_prompt_with_tinyllama(base_prompt)


In [7]:
def initialize_system():
    """Initialize the entire prompt engineering system"""
    print("Initializing Prompt Engineering System...")

    # Load prompts data
    df = load_prompts_data()
    if df is None:
        return False

    # Setup vector database
    success = setup_vector_database(df)
    if not success:
        return False

    print("✅ System initialized successfully!")
    return True

def demonstrate_system():
    """Demonstrate the prompt engineering system with sample queries"""
    test_queries = [
        "I need a poem about the beauty of nature.",
        "Tell me a story about a lion and a mouse that teaches a lesson.",
        "Give me a prompt for creating an image of a futuristic city.",
        "Help me create a short story for children about a little bird learning to fly.",
        "Suggest a way to generate a business plan for an AI startup."
    ]

    print("\n🔍 Demonstrating the prompt engineering system:\n")
    for query in test_queries:
        print(f"User query: {query}")

        # Step 1: Generate base prompt
        base_prompt = generate_custom_prompt(query)
        print(f"Base prompt: {base_prompt}")

        # Step 2: Enhance with TinyLlama
        enhanced_prompt = enhance_prompt_with_tinyllama(base_prompt)
        print(f"Enhanced prompt: {enhanced_prompt}")
        print("-" * 80)


In [11]:
pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.3 MB/s eta 0:00:00


In [8]:
def create_web_interface():
    """Create a Gradio web interface for the prompt engineering system"""
    try:
        import gradio as gr

        def process_query(query):
            base_prompt = generate_custom_prompt(query)
            enhanced_prompt = enhance_prompt_with_tinyllama(base_prompt)
            return base_prompt, enhanced_prompt

        with gr.Blocks(title="Prompt Engineering System") as demo:
            gr.Markdown("# 🧠 Prompt Engineering System")
            gr.Markdown("Enter your query, and the system will generate a customized prompt for you.")

            with gr.Row():
                query_input = gr.Textbox(label="Your Query", placeholder="e.g., I need a poem about nature")

            with gr.Row():
                submit_btn = gr.Button("Generate Prompt")

            with gr.Row():
                base_output = gr.Textbox(label="Base Prompt")
                enhanced_output = gr.Textbox(label="Enhanced Prompt")

            submit_btn.click(
                fn=process_query,
                inputs=query_input,
                outputs=[base_output, enhanced_output]
            )

        return demo
    except ImportError:
        print("Gradio not installed. Run 'pip install gradio' to enable the web interface.")
        return None

if __name__ == "__main__":
    # Initialize the system
    if initialize_system():
        # Demonstrate the system
        demonstrate_system()

        # Create and launch web interface
        demo = create_web_interface()
        if demo:
            demo.launch()
    else:
        print("❌ System initialization failed. Please check the error messages above.")

Initializing Prompt Engineering System...
✅ Loaded 213 prompts from the dataset
Loading embedding model...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating embeddings (this may take a while)...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Generated 213 embeddings of dimension 384
✅ Dropped existing collection: prompt_builder_agent
✅ Created collection: prompt_builder_agent
✅ Inserted 213 records into the collection
✅ Created vector index
✅ Collection loaded and ready for querying
✅ System initialized successfully!

🔍 Demonstrating the prompt engineering system:

User query: I need a poem about the beauty of nature.
Base prompt: I want you to act as a Poet. I want you to act as a poet. You will create poems that evoke emotions and have the power to stir people's soul. Write on any topic or theme but make sure your words convey the feeling you are trying to express in beautiful yet meaningful ways. You can also come up with short verses that are still powerful enough to leave an imprint in readers' minds. Focus specifically on the beauty of nature. My request is: I need a poem about the beauty of nature.
Enhanced prompt: I want you to act as a Poet. I want you to act as a poet. You will create poems that evoke emotions 

In [ ]:
# Import additional required libraries
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Define model paths
TINYLLAMA_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
PHI2_MODEL = "microsoft/phi-2"

# Function to enhance prompts with Phi-2
def enhance_prompt_with_phi2(base_prompt):
    """Enhance a prompt using the Microsoft Phi-2 model"""
    try:
        # Load Phi-2 model and tokenizer
        print("Loading Phi-2 model...")
        tokenizer = AutoTokenizer.from_pretrained(PHI2_MODEL)
        model = AutoModelForCausalLM.from_pretrained(PHI2_MODEL)

        # Move model to GPU if available
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)

        # Prepare input
        prompt = f"Improve the following prompt for clarity and creativity:\n\n{base_prompt}\n\nEnhanced prompt:"
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        # Generate enhanced prompt
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode and extract enhanced prompt
        result = tokenizer.decode(outputs[0], skip_special_tokens=True)
        enhanced = result.split("Enhanced prompt:")[-1].strip()

        print("Phi-2 enhancement complete!")
        return enhanced
    except Exception as e:
        print(f"❌ Phi-2 enhancement failed: {str(e)}")
        return base_prompt  # Fallback to base prompt

# Test function for Phi-2
def test_phi2_enhancement():
    """Test the Phi-2 enhancement function"""
    test_prompt = "I need help writing a story about a magical forest."
    print("Testing Phi-2 enhancement...")
    print(f"Test prompt: {test_prompt}")
    enhanced = enhance_prompt_with_phi2(test_prompt)
    print(f"Enhanced prompt: {enhanced}")
    return enhanced

# Updated Gradio interface with model selection
def create_web_interface_with_model_choice():
    """Create a Gradio web interface with model selection"""
    try:
        import gradio as gr

        def process_query(query, model_choice):
            print(f"Processing query with {model_choice} model...")

            # Generate base prompt
            base_prompt = generate_custom_prompt(query)

            # Enhance with selected model
            if model_choice == "Phi-2":
                enhanced_prompt = enhance_prompt_with_phi2(base_prompt)
            else:  # Default to TinyLlama
                enhanced_prompt = enhance_prompt_with_tinyllama(base_prompt)

            return enhanced_prompt

        # Custom CSS for better appearance
        custom_css = """
        .gradio-container {
            background-color: #f9f9f9 !important;
        }
        .title {
            color: #2a6099 !important;
            text-align: center;
        }
        .subtitle {
            color: #555 !important;
            text-align: center;
            margin-bottom: 20px;
        }
        .gr-button-primary {
            background-color: #2a6099 !important;
            border-color: #2a6099 !important;
        }
        .footer {
            text-align: center;
            margin-top: 20px;
            color: #666;
            font-size: 0.9em;
        }
        """

        with gr.Blocks(title="AI Prompt Engineer", css=custom_css) as demo:
            gr.HTML("<h1 class='title'>🧠 AI Prompt Engineer</h1>")
            gr.HTML("<p class='subtitle'>Transform simple ideas into powerful AI prompts</p>")

            with gr.Row():
                query_input = gr.Textbox(
                    label="What would you like help with?",
                    placeholder="e.g., I need a creative story about a space explorer",
                    lines=3
                )

            with gr.Row():
                model_choice = gr.Radio(
                    ["TinyLlama", "Phi-2"],
                    label="Select Enhancement Model",
                    value="TinyLlama"
                )

            with gr.Row():
                submit_btn = gr.Button("Generate Enhanced Prompt", variant="primary")

            with gr.Row():
                enhanced_output = gr.Textbox(
                    label="Your AI-Optimized Prompt",
                    lines=8,
                    show_copy_button=True
                )

            # Examples to help users get started
            gr.Examples(
                examples=[
                    "I need a poem about the beauty of nature",
                    "Create a business plan for an AI startup",
                    "Help me write a story about a detective solving a mystery",
                    "Design a prompt for generating images of futuristic cities"
                ],
                inputs=query_input
            )

            gr.HTML("<div class='footer'>Powered by vector similarity search with TinyLlama & Phi-2</div>")

            submit_btn.click(
                fn=process_query,
                inputs=[query_input, model_choice],
                outputs=enhanced_output
            )

        return demo
    except ImportError:
        print("Gradio not installed. Run 'pip install gradio' to enable the web interface.")
        return None

# Function to test both models with the same query
def compare_models():
    """Compare TinyLlama and Phi-2 enhancement on the same query"""
    test_query = "I need a creative story about aliens visiting Earth"

    print("\n====== MODEL COMPARISON TEST ======")
    print(f"Test query: {test_query}")

    # Generate base prompt
    base_prompt = generate_custom_prompt(test_query)
    print(f"\nBase prompt: {base_prompt}")

    # Enhance with TinyLlama
    print("\n--- TinyLlama Enhancement ---")
    tinyllama_enhanced = enhance_prompt_with_tinyllama(base_prompt)
    print(f"TinyLlama result: {tinyllama_enhanced}")

    # Enhance with Phi-2
    print("\n--- Phi-2 Enhancement ---")
    phi2_enhanced = enhance_prompt_with_phi2(base_prompt)
    print(f"Phi-2 result: {phi2_enhanced}")

    print("\n====== COMPARISON COMPLETE ======")

# Updated script execution
if __name__ == "__main__":
    # Initialize the system
    if initialize_system():
        # Optional: Test Phi-2 model
        test_phi2_enhancement()

        # Optional: Compare both models
        compare_models()

        # Create and launch web interface with model selection
        demo = create_web_interface_with_model_choice()
        if demo:
            demo.launch(share=True)  # Add share=True for a public URL
    else:
        print("❌ System initialization failed. Please check the error messages above.")

Initializing Prompt Engineering System...
✅ Loaded 213 prompts from the dataset
Loading embedding model...
Generating embeddings (this may take a while)...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Generated 213 embeddings of dimension 384
✅ Dropped existing collection: prompt_builder_agent
✅ Created collection: prompt_builder_agent
✅ Inserted 213 records into the collection
✅ Created vector index
✅ Collection loaded and ready for querying
✅ System initialized successfully!
Testing Phi-2 enhancement...
Test prompt: I need help writing a story about a magical forest.
Loading Phi-2 model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]